### Imports and intialization


In [1]:
import ee
import geemap

# Initialize with your Project ID
ee.Initialize(project='solar-geoai-dferreri45')

# 1. Define the Site: Near Postmasburg, Northern Cape

#### Coordinates: -28.3N, 23.3E (High solar potential area)


In [2]:
point = ee.Geometry.Point([23.3, -28.3])
roi = point.buffer(20000) # 20km radius

# 2. Setup the Sentinel-2 Pipeline


In [3]:
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000) # Scale to 0-1

# 3. Pull and Process


In [4]:
s2_collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate('2025-01-01', '2025-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds))

## Create the Median Composite


In [5]:
composite = s2_collection.median().clip(roi)


# 4. Pull Elevation Data (SRTM)


In [6]:
srtm = ee.Image("USGS/SRTMGL1_003").clip(roi)
slope = ee.Terrain.slope(srtm)

# 5. Visualization


In [7]:
Map = geemap.Map()
Map.centerObject(point, 11)

### True Color (Visual check)


In [8]:
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'Satellite (True Color)')

### Slope (Engineering check - Red is steep, Green is flat)


In [9]:
Map.addLayer(slope, {'min': 0, 'max': 10, 'palette': ['green', 'yellow', 'red']}, 'Terrain Slope')

### Plot


In [10]:
Map

Map(center=[-28.299999999999994, 23.3], controls=(WidgetControl(options=['position', 'transparent_bg'], positi…